In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import from intervalinf
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue
from intervalinf.operators.gradient import Gradient

# Set up plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Creating a Gradient Operator on L² Space

### What is the Gradient Operator?

The **gradient operator** (or derivative in 1D) is defined as:

$$
\nabla: f(x) \mapsto \frac{df}{dx}(x)
$$

When defined on the **Lebesgue space** $L^2([a,b])$, the gradient is an **unbounded operator**. This means:

1. **Not defined everywhere**: The gradient doesn't exist for all functions in $L^2$
   - Example: Step functions are in $L^2$ but don't have derivatives
   
2. **Domain is dense but not complete**: The natural domain is smooth functions, which is dense in $L^2$ but not all of $L^2$

3. **No continuity bound**: There's no constant $C$ such that $\|\nabla f\|_{L^2} \leq C \|f\|_{L^2}$ for all $f$
   - Functions can be small in $L^2$ but have large derivatives
   - Example: $f_n(x) = \frac{1}{\sqrt{n}} \sin(nx)$ has $\|f_n\|_{L^2} \to 0$ but $\|\nabla f_n\|_{L^2} \to \infty$

### Practical Implications

When using the gradient operator on a Lebesgue space:

- ✅ **It will work** if we give it smooth, well-behaved functions
- ⚠️ **We must be careful** about function regularity
- ❌ **It will fail or give nonsense** for rough/discontinuous functions
- 🔧 **Numerical approximation** (finite differences) may mask mathematical issues

**The correct setting** for differential operators is **Sobolev spaces** $H^s$, where the gradient becomes a bounded operator (we'll preview this at the end).

### Core API Usage:

In [ ]:
# Create domain and Lebesgue space
domain = IntervalDomain(0, 1)
L2 = Lebesgue(0, domain, basis=None)

# Create gradient operator
grad = Gradient(L2, fd_order=2, fd_step=None)  # Auto-compute step size

# Create a smooth test function
f = Function(L2, evaluate_callable=lambda x: x**2, name="f(x) = x²")

# Apply gradient
df = grad(f)

# Evaluate at a point
x_test = 0.5
print(f"f({x_test}) = {f.evaluate(x_test):.4f}")
print(f"df/dx({x_test}) = {df.evaluate(x_test):.4f}")
print(f"Expected (2x): {2*x_test:.4f}")

### Creating the Gradient Operator:

In [ ]:
# Create domain and Lebesgue space
domain = IntervalDomain(0, 1)
L2 = Lebesgue(20, domain, basis='fourier')

print(f"Created L² space:")
print(f"  Domain: [{L2.function_domain.a}, {L2.function_domain.b}]")
print(f"  Dimension: {L2.dim}")
print(f"  Basis: Fourier")
print()

# Create gradient operator
grad = Gradient(L2, fd_order=2, fd_step=None)  # Auto-compute step size

print("Created Gradient operator:")
print(f"  Method: Finite differences")
print(f"  Order: 2 (second-order accurate)")
print(f"  Step size: Auto (computed from domain)")
print()
print("⚠️  Warning: This is an UNBOUNDED operator on L²!")
print("   It only makes sense for smooth functions.")

## 2. Success Case: Smooth Functions

When we apply the gradient to **smooth functions**, everything works beautifully.

### Core API Usage:

In [ ]:
# Create a smooth function
f = Function(L2, evaluate_callable=lambda x: np.sin(2*np.pi*x), name="sin(2πx)")

# Apply gradient (should give 2π·cos(2πx))
df = grad(f)

# Test at a point
x = 0.25
computed = df.evaluate(x)
expected = 2*np.pi*np.cos(2*np.pi*x)

print(f"f(x) = sin(2πx)")
print(f"At x = {x}:")
print(f"  Computed: {computed:.6f}")
print(f"  Expected: {expected:.6f}")
print(f"  Error:    {abs(computed - expected):.2e}")

### Example: Differentiating Polynomials and Trigonometric Functions

In [ ]:
# Create smooth test functions with known derivatives
test_cases = [
    {
        'name': 'f(x) = x²',
        'function': lambda x: x**2,
        'derivative': lambda x: 2*x,
        'test_point': 0.5
    },
    {
        'name': 'f(x) = sin(2πx)',
        'function': lambda x: np.sin(2*np.pi*x),
        'derivative': lambda x: 2*np.pi*np.cos(2*np.pi*x),
        'test_point': 0.25
    },
    {
        'name': 'f(x) = x³ - 3x² + 2x',
        'function': lambda x: x**3 - 3*x**2 + 2*x,
        'derivative': lambda x: 3*x**2 - 6*x + 2,
        'test_point': 0.7
    },
]

print("Testing gradient on smooth functions:")
print("="*70)

for case in test_cases:
    # Create function in L²
    f = Function(L2, evaluate_callable=case['function'], name=case['name'])

    # Apply gradient
    df = grad(f)

    # Evaluate at test point
    x = case['test_point']
    computed = df.evaluate(x)
    expected = case['derivative'](x)
    error = abs(computed - expected)

    print(f"\n{case['name']}:")
    print(f"  At x = {x}:")
    print(f"    Computed: {computed:.8f}")
    print(f"    Expected: {expected:.8f}")
    print(f"    Error:    {error:.2e}")

    if error < 1e-4:
        print(f"    ✓ Excellent agreement!")
    else:
        print(f"    ⚠ Noticeable error (expected for finite differences)")

### Visualization: Smooth Function Differentiation

In [ ]:
# Visualize the gradient for smooth functions
x = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, case in enumerate(test_cases):
    # Create function
    f = Function(L2, evaluate_callable=case['function'])
    df = grad(f)

    # Evaluate
    f_vals = f.evaluate(x)
    df_vals = df.evaluate(x)
    df_exact = case['derivative'](x)

    # Plot
    ax = axes[i]
    ax.plot(x, f_vals, 'b-', linewidth=2, label='f(x)', alpha=0.7)
    ax.plot(x, df_vals, 'r-', linewidth=2, label="∇f (computed)", alpha=0.7)
    ax.plot(x, df_exact, 'g--', linewidth=1, label="df/dx (exact)", alpha=0.5)
    ax.set_xlabel('x')
    ax.set_title(case['name'])
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ For smooth functions, the finite difference gradient works excellently!")

## 3. Failure Case: Discontinuous Functions

Now let's see what happens when we apply the gradient to **rough or discontinuous** functions that are still in $L^2$.

### Core API Usage:

In [ ]:
# Create a step function (discontinuous but in L²)
def step_function(x):
    """Step function: f(x) = 0 for x < 0.5, f(x) = 1 for x >= 0.5"""
    return np.where(np.asarray(x) < 0.5, 0.0, 1.0)

f_step = Function(L2, evaluate_callable=step_function, name="Step function")

# Try to apply gradient
df_step = grad(f_step)

# Evaluate at the discontinuity
x_disc = 0.5
print(f"Step function at x = {x_disc}: {f_step.evaluate(x_disc)}")
print(f"'Gradient' at x = {x_disc}: {df_step.evaluate(x_disc)}")
print()
print("⚠️  WARNING: This result is NOT mathematically valid!")
print("   The derivative at a discontinuity doesn't exist.")

### Example: Step Function (Discontinuous)

In [ ]:
# Create a step function (discontinuous but in L²)
def step_function(x):
    """Step function: f(x) = 0 for x < 0.5, f(x) = 1 for x >= 0.5"""
    return np.where(np.asarray(x) < 0.5, 0.0, 1.0)

# Create function in L²
f_step = Function(L2, evaluate_callable=step_function, name="Step function")

print("Step Function Properties:")
print(f"  Discontinuous at x = 0.5")
print(f"  In L²? YES (square-integrable)")
print(f"  Has derivative? NO (mathematically)")
print(f"  Derivative: δ(x - 0.5) (Dirac delta - not a function!)")
print()

# Try to apply gradient
df_step = grad(f_step)

# Evaluate the "derivative"
x_eval = np.linspace(0, 1, 500)
f_vals = f_step.evaluate(x_eval)
df_vals = df_step.evaluate(x_eval)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original function
ax1.plot(x_eval, f_vals, 'b-', linewidth=2)
ax1.axvline(0.5, color='r', linestyle='--', alpha=0.5, label='Discontinuity')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.set_title('Step Function (in L²)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# "Derivative" computed by finite differences
ax2.plot(x_eval, df_vals, 'r-', linewidth=2)
ax2.axvline(0.5, color='r', linestyle='--', alpha=0.5, label='Discontinuity location')
ax2.set_xlabel('x')
ax2.set_ylabel('∇f (computed)')
ax2.set_title('Finite Difference "Gradient" (NOT MATHEMATICALLY VALID)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️  WARNING: What we computed is NOT a valid derivative!")
print("   - The mathematical derivative doesn't exist (it's a Dirac delta)")
print("   - Finite differences give a numerical approximation of the jump")
print("   - This is NOT in L² (it's a distribution)")
print("   - Using this result in further computations will lead to errors")

### Example: Piecewise Linear Function (Continuous but Non-Smooth)

In [ ]:
# Create a continuous but non-smooth function (piecewise linear)
def piecewise_linear(x):
    """V-shaped function: f(x) = |x - 0.5|"""
    return np.abs(np.asarray(x) - 0.5)

f_pwl = Function(L2, evaluate_callable=piecewise_linear, name="Piecewise linear")

print("Piecewise Linear Function Properties:")
print(f"  Continuous everywhere: YES")
print(f"  In L²: YES")
print(f"  Smooth at x = 0.5? NO (kink)")
print(f"  Has derivative everywhere? NO (undefined at x = 0.5)")
print()

# Apply gradient
df_pwl = grad(f_pwl)

# Evaluate
x_eval = np.linspace(0, 1, 500)
f_vals = f_pwl.evaluate(x_eval)
df_vals = df_pwl.evaluate(x_eval)

# Exact derivative (where it exists)
def exact_derivative(x):
    return np.where(np.asarray(x) < 0.5, -1.0, 1.0)

df_exact = exact_derivative(x_eval)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x_eval, f_vals, 'b-', linewidth=2)
ax1.axvline(0.5, color='r', linestyle='--', alpha=0.5, label='Non-smooth point')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.set_title('Piecewise Linear Function (|x - 0.5|)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(x_eval, df_vals, 'r-', linewidth=2, label='Computed (finite diff)')
ax2.plot(x_eval, df_exact, 'g--', linewidth=1, alpha=0.5, label='Exact (where exists)')
ax2.axvline(0.5, color='r', linestyle='--', alpha=0.5)
ax2.set_xlabel('x')
ax2.set_ylabel('∇f')
ax2.set_title('Gradient (undefined at kink)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([-1.5, 1.5])

plt.tight_layout()
plt.show()

print("\n⚠️  At the kink (x = 0.5):")
print(f"   Finite difference approximation: {df_pwl.evaluate(0.5):.6f}")
print(f"   Mathematical derivative: UNDEFINED")
print(f"   The result is an artifact of the numerical method!")

## 4. The Role of Function Regularity

Let's quantify how function smoothness affects gradient accuracy.

### Core API Usage:

In [ ]:
# Test with functions of increasing frequency
k = 5
f = Function(L2, evaluate_callable=lambda x, freq=k: np.sin(2*np.pi*freq*x))
df = grad(f)

# Compute error
x_test = np.linspace(0, 1, 1000)
df_computed = df.evaluate(x_test)
df_exact = 2*np.pi*k*np.cos(2*np.pi*k*x_test)
error = np.sqrt(np.trapezoid((df_computed - df_exact)**2, x_test))

print(f"Function: sin(2π·{k}·x)")
print(f"L² error in gradient: {error:.2e}")

### Experiment: Functions with Increasing Frequency

In [ ]:
# Create functions with increasing frequency: f_k(x) = sin(2πkx)
# As k increases, functions oscillate faster and are "rougher"

frequencies = [1, 2, 5, 10, 20]
step_size = (domain.b - domain.a) / 1000  # Fixed step size for fair comparison

results = []

print("Testing gradient accuracy vs. function roughness:")
print("="*70)

for k in frequencies:
    # Create function
    f = Function(L2, evaluate_callable=lambda x, freq=k: np.sin(2*np.pi*freq*x))
    df_exact_func = lambda x, freq=k: 2*np.pi*freq*np.cos(2*np.pi*freq*x)

    # Apply gradient
    df = grad(f)

    # Compute L² error
    x_dense = np.linspace(0, 1, 1000)
    df_computed = df.evaluate(x_dense)
    df_exact = df_exact_func(x_dense)

    error_L2 = np.sqrt(np.trapezoid((df_computed - df_exact)**2, x_dense))

    # Relative error
    norm_exact = np.sqrt(np.trapezoid(df_exact**2, x_dense))
    relative_error = error_L2 / norm_exact

    results.append({
        'frequency': k,
        'error_L2': error_L2,
        'relative_error': relative_error
    })

    print(f"\nFrequency k = {k:2d}:")
    print(f"  Function: sin(2π·{k}·x)")
    print(f"  L² error: {error_L2:.2e}")
    print(f"  Relative error: {relative_error:.2%}")

# Plot error vs frequency
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

freqs = [r['frequency'] for r in results]
errors_abs = [r['error_L2'] for r in results]
errors_rel = [r['relative_error'] for r in results]

ax1.semilogy(freqs, errors_abs, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Frequency k')
ax1.set_ylabel('L² Error')
ax1.set_title('Absolute Error vs. Function Roughness')
ax1.grid(True, alpha=0.3)

ax2.plot(freqs, [e*100 for e in errors_rel], 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Frequency k')
ax2.set_ylabel('Relative Error (%)')
ax2.set_title('Relative Error vs. Function Roughness')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("📊 Observation: Error increases with function roughness!")
print("   As functions oscillate faster, finite differences become less accurate.")

## 5. Comparing Different Finite Difference Orders

Higher-order finite difference schemes can improve accuracy for smooth functions.

### Core API Usage:

In [ ]:
# Compare 2nd and 4th order finite differences
grad_2nd = Gradient(L2, fd_order=2)
grad_4th = Gradient(L2, fd_order=4)

# Test function
f = Function(L2, evaluate_callable=lambda x: np.sin(2*np.pi*x))

# Apply both gradients
df_2nd = grad_2nd(f)
df_4th = grad_4th(f)

# Check errors at a point
x = 0.25
exact = 2*np.pi*np.cos(2*np.pi*x)
error_2nd = abs(df_2nd.evaluate(x) - exact)
error_4th = abs(df_4th.evaluate(x) - exact)

print(f"At x = {x}:")
print(f"  2nd order error: {error_2nd:.2e}")
print(f"  4th order error: {error_4th:.2e}")
print(f"  Improvement: {error_2nd/error_4th:.1f}x")

### Detailed Comparison

In [ ]:
# Compare 2nd and 4th order finite differences
grad_2nd = Gradient(L2, fd_order=2)
grad_4th = Gradient(L2, fd_order=4)

# Test function: f(x) = sin(2πx)
f = Function(L2, evaluate_callable=lambda x: np.sin(2*np.pi*x))
df_exact_func = lambda x: 2*np.pi*np.cos(2*np.pi*x)

# Apply both gradients
df_2nd = grad_2nd(f)
df_4th = grad_4th(f)

# Evaluate
x_eval = np.linspace(0, 1, 500)
f_vals = f.evaluate(x_eval)
df_2nd_vals = df_2nd.evaluate(x_eval)
df_4th_vals = df_4th.evaluate(x_eval)
df_exact = df_exact_func(x_eval)

# Compute errors
error_2nd = np.abs(df_2nd_vals - df_exact)
error_4th = np.abs(df_4th_vals - df_exact)

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Function and derivatives
ax = axes[0, 0]
ax.plot(x_eval, f_vals, 'b-', linewidth=2, label='f(x)')
ax.plot(x_eval, df_exact, 'g--', linewidth=1, alpha=0.5, label='Exact derivative')
ax.set_xlabel('x')
ax.set_ylabel('Value')
ax.set_title('Original Function: f(x) = sin(2πx)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2nd order
ax = axes[0, 1]
ax.plot(x_eval, df_2nd_vals, 'r-', linewidth=2, label='2nd order FD')
ax.plot(x_eval, df_exact, 'g--', linewidth=1, alpha=0.5, label='Exact')
ax.set_xlabel('x')
ax.set_ylabel('df/dx')
ax.set_title('2nd Order Finite Difference')
ax.legend()
ax.grid(True, alpha=0.3)

# 4th order
ax = axes[1, 0]
ax.plot(x_eval, df_4th_vals, 'purple', linewidth=2, label='4th order FD')
ax.plot(x_eval, df_exact, 'g--', linewidth=1, alpha=0.5, label='Exact')
ax.set_xlabel('x')
ax.set_ylabel('df/dx')
ax.set_title('4th Order Finite Difference')
ax.legend()
ax.grid(True, alpha=0.3)

# Error comparison
ax = axes[1, 1]
ax.semilogy(x_eval, error_2nd, 'r-', linewidth=2, label='2nd order error')
ax.semilogy(x_eval, error_4th, 'purple', linewidth=2, label='4th order error')
ax.set_xlabel('x')
ax.set_ylabel('Absolute Error')
ax.set_title('Error Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

max_error_2nd = np.max(error_2nd)
max_error_4th = np.max(error_4th)

print(f"Maximum Errors:")
print(f"  2nd order: {max_error_2nd:.2e}")
print(f"  4th order: {max_error_4th:.2e}")
print(f"  Improvement: {max_error_2nd/max_error_4th:.1f}x")
print()
print("✓ Higher-order schemes give better accuracy for smooth functions!")

## 6. Preview: Bounded Operators on Sobolev Spaces

The **proper mathematical setting** for differential operators is **Sobolev spaces** $H^s([a,b])$.

### Why Sobolev Spaces?

In a Sobolev space $H^s$ with $s \geq 1$, the gradient becomes a **bounded operator**:

$$
\nabla: H^s([a,b]) \to H^{s-1}([a,b])
$$

This means:
- ✓ **Well-defined** on the entire space
- ✓ **Continuous**: $\|\nabla f\|_{H^{s-1}} \leq C \|f\|_{H^s}$ for some constant $C$
- ✓ **Stable**: Small changes in $f$ lead to small changes in $\nabla f$

### Mathematical Intuition

Sobolev spaces $H^s$ are built using the Laplacian operator to "measure smoothness":

$$
H^s = \text{domain of } (k^2 I + \Delta)^{s/2}
$$

Functions in $H^s$ have $s$ derivatives in $L^2$, making differential operators bounded.

### Core API Usage:

In [ ]:
# For a proper Sobolev space setup, we would need:
# 1. A Laplacian operator on L²
# 2. Use it to construct H^s
# 3. Then the gradient on H^s is bounded

print("Proper workflow for bounded differential operators:")
print()
print("1. L² space (ambient)")
print("   L2 = Lebesgue(dim, domain)")
print()
print("2. Laplacian on L² (unbounded operator)")
print("   Δ = Laplacian(L2, boundary_conditions)")
print()
print("3. Sobolev space H^s via Laplacian spectrum")
print("   H1 = Sobolev(dim, domain, s=1, k=1, L=Δ)")
print()
print("4. Gradient on H^s (NOW BOUNDED!)")
print("   ∇ = Gradient(H1)")
print()
print("Result: ∇: H¹ → L² is continuous and stable!")
print()
print("📚 See the Sobolev spaces demo for full details.")

## 7. Summary and Best Practices

### Key Takeaways

1. **The gradient on L² is unbounded**
   - Not defined for all functions in L²
   - No continuity estimate
   - Mathematically subtle!

2. **It works well for smooth functions**
   - Polynomials, trigonometric functions, exponentials
   - Finite differences give good approximations
   - Higher-order schemes → better accuracy

3. **It fails for rough functions**
   - Step functions: derivative is a Dirac delta (not a function)
   - Piecewise linear: derivative undefined at kinks
   - Results are numerical artifacts, not mathematics

4. **Function regularity matters**
   - Smoother functions → more accurate gradients
   - High-frequency oscillations → larger errors
   - Trade-off between resolution and accuracy

### Best Practices When Using Gradient on L²

✅ **DO:**
- Use it only for functions you know are smooth
- Check that your functions are continuously differentiable
- Validate results against known derivatives
- Use higher-order finite differences for better accuracy
- Be aware of boundary treatment

⚠️ **DON'T:**
- Apply it blindly to arbitrary L² functions
- Trust results for discontinuous or kinked functions
- Assume the operator is continuous/bounded
- Use it in critical applications without validation

### The Right Tool for the Job

For rigorous treatment of differential operators:

**Use Sobolev spaces $H^s$!**
- Gradient is bounded: $\nabla: H^s \to H^{s-1}$
- Well-defined theory
- Stable numerical methods
- Proper mathematical framework

### Next Steps

- Explore **Sobolev spaces** for bounded differential operators
- Learn about **Laplacian operators** and spectral methods
- Study **variational problems** and weak derivatives
- Investigate **operator theory** and functional analysis